In [ ]:
import pandas as pd
import sys
import os

# ── Paths & sys.path ─────────────────────────────────────────────────────────
# In Jupyter notebooks, __file__ is not defined. We use os.getcwd() instead.
SCRIPT_DIR = os.getcwd()
sys.path.insert(0, SCRIPT_DIR)
sys.path.insert(0, os.path.abspath(os.path.join(SCRIPT_DIR, '..')))

from utils import compute_similarities_allvsall, compute_distances_allvsall
    
from plots import plot_networkx_plotly, plot_threshold_network

In [2]:
# 1. Load your dataset (assuming it has 'date', 'product_id', and 'sales')
val_size_global        = 60
forecast_horizon_global = 153
train_size_global       = 761 - val_size_global - forecast_horizon_global  # 455

_HERE     = os.getcwd()
DATA_PATH = os.path.join(_HERE, "../../../dataset/data_andre_fulfilled.feather")
TOP_PATH  = os.path.join(_HERE, "../../../dataset/top_12500.feather")

# Load the dataset
df = pd.read_feather(DATA_PATH)

# 2. Pivot the data to get shape (N products, T timesteps)
# We fill NaNs with 0 (assuming no sales means 0)
ts_matrix_df = df.pivot(index='item_id', columns='date', values='value')

# Extract the N x T numpy array
all_ts = ts_matrix_df.values

# Extract the node labels (product IDs) to label our graph
product_ids = ts_matrix_df.index.tolist()

print(f"Data shape: {all_ts.shape} -> ({len(product_ids)} products over {all_ts.shape[1]} timesteps)")

Data shape: (1427, 761) -> (1427 products over 761 timesteps)


In [3]:
# Choose your metric ('spearman', 'pearson', or 'kendall')
metric_used = 'spearman'

# Compute the N x N similarity matrix
similarity_matrix = compute_similarities_allvsall(all_ts, metric=metric_used)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

Similarity matrix shape: (1427, 1427)


In [6]:


# Set your threshold (e.g., only connect products with a Spearman correlation >= 0.65)
tau = 0.6

fig = plot_threshold_network(
    matrix=similarity_matrix,
    node_labels=product_ids,
    threshold=tau,
    metric_type='similarity',
    title=f"Retail Supply Chain Graph ({metric_used.capitalize()} \u2265 {tau})"
)

fig.show()

In [7]:
# Pivot the data and fill NaNs with 0
ts_matrix_df = df.pivot(index='item_id', columns='date', values='value').fillna(0)
product_ids = ts_matrix_df.index.tolist()

# ---------------------------------------------------------
# 3. FAST VECTORIZED Z-NORMALIZATION (Fit on Train only)
# ---------------------------------------------------------
print("Applying Vectorized Z-score normalisation...")
L = ts_matrix_df.shape[1]
global_val_start_idx = L - forecast_horizon_global - val_size_global
global_train_start_idx = max(0, global_val_start_idx - train_size_global)

# Isolate the training slice (Time steps: 0 to end of train)
train_slice = ts_matrix_df.iloc[:, global_train_start_idx:global_val_start_idx]

# Compute means and standard deviations strictly on the train slice
train_means = train_slice.mean(axis=1)
train_stds  = train_slice.std(axis=1)

# Prevent division by zero for flat products (e.g., items with 0 sales in the train window)
train_stds = train_stds.replace(0, 1e-8)

# Apply the transformation to the ENTIRE dataframe instantaneously
ts_matrix_df_scaled = ts_matrix_df.sub(train_means, axis=0).div(train_stds, axis=0)

# Extract the N x T numpy array from the SCALED dataframe
all_ts_scaled = ts_matrix_df_scaled.values

print(f"Scaled Data shape: {all_ts_scaled.shape} -> ({len(product_ids)} products over {all_ts_scaled.shape[1]} timesteps)")

# ---------------------------------------------------------
# 4. Compute Distance Matrix
# ---------------------------------------------------------
# Choose your metric
metric_used = 'cid'

# Compute the N x N distance matrix using the scaled data
distance_matrix = compute_distances_allvsall(all_ts_scaled, metric=metric_used)

print(f"Distance matrix shape: {distance_matrix.shape}")

Applying Vectorized Z-score normalisation...
Scaled Data shape: (1427, 761) -> (1427 products over 761 timesteps)


NameError: name 'compute_distances_allvsall' is not defined